In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [3]:
df.shape

(569, 33)

In [4]:
df.drop(columns=['id','Unnamed: 32'] ,inplace = True )

In [5]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [6]:
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler , LabelEncoder

In [7]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

In [8]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [9]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [10]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [11]:
X_train_tensor.shape , y_train_tensor.shape

(torch.Size([455, 30]), torch.Size([455]))

In [12]:
class MySimpleNN() :
    def __init__(self, X):
        self.weights = torch.rand(X.shape[1],1 ,dtype=torch.float64 ,requires_grad = True ) #tensor of shape(cols,1)
        self.bias = torch.zeros(1 , dtype = torch.float32 , requires_grad = True)
    def forward(self,X):
        z = torch.matmul(X,self.weights) + self.bias
        y_pred = torch.sigmoid(z)
        return y_pred
    def loss_func(self,y_pred ,y):
        #clamp pred to avoid log(0)
        epsilon = 1e-7
        y_pred = torch.clamp(y_pred , epsilon , 1-epsilon)
        loss = -(y_train_tensor * torch.log(y_pred) +(1-y_train_tensor)*torch.log(1-y_pred)).mean()
        return loss

In [13]:
learning_rate = 0.1
epochs = 25

In [14]:
model = MySimpleNN(X_train_tensor)

for epoch in range(epochs):
    y_pred = model.forward(X_train_tensor)
    loss = model.loss_func(y_pred , y_train_tensor)
    loss.backward()
    with torch.no_grad() : #parameters update #to_grad -> to not track gradients 1.e to not make it part of computation graoh
        model.weights -= learning_rate * model.weights.grad
        model.bias -= learning_rate * model.bias.grad

    model.weights.grad.zero_()
    model.bias.grad.zero_()
    print(f"Epoch:{epoch+1},Loss:{loss}")

Epoch:1,Loss:3.21276066213219
Epoch:2,Loss:3.080034797667335
Epoch:3,Loss:2.93851342645307
Epoch:4,Loss:2.7931275253273573
Epoch:5,Loss:2.6400185712166917
Epoch:6,Loss:2.4831401007811142
Epoch:7,Loss:2.3215604378624097
Epoch:8,Loss:2.1607801572716268
Epoch:9,Loss:2.003831362974157
Epoch:10,Loss:1.8537347205534627
Epoch:11,Loss:1.7105699201244058
Epoch:12,Loss:1.5741661724428448
Epoch:13,Loss:1.4480226675715089
Epoch:14,Loss:1.3308520661876972
Epoch:15,Loss:1.2218646258882215
Epoch:16,Loss:1.1262895903312105
Epoch:17,Loss:1.0473762030997966
Epoch:18,Loss:0.9839338459232906
Epoch:19,Loss:0.9340956007035485
Epoch:20,Loss:0.8956200292368797
Epoch:21,Loss:0.8661596912088901
Epoch:22,Loss:0.8435031406962838
Epoch:23,Loss:0.8257663770554692
Epoch:24,Loss:0.8114857091838951
Epoch:25,Loss:0.7996070960306004


In [15]:
with torch.no_grad():
    y_pred = model.forward(X_test_tensor)
    y_pred = (y_pred > 0.9).float()
    accuracy = (y_pred == y_test_tensor).float().mean()
    print(f"Accuracy:{accuracy}")

Accuracy:0.604032039642334
